# **Role-Based Job Market Analysis: Skills, Demand & Compensation Insights**

## This notebook analyzes 97k+ job listings across India to uncover trends in roles, skills, and salaries. It provides actionable insights on skill demand, compensation patterns, and hiring trends using visualizations and aggregated metrics.

 ### **Step 1:** Importing the Libraries

In [ ]:
# Import pandas for data manipulation and analysis
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for plotting
import matplotlib.pyplot as plt

# Import seaborn for advanced statistical visualization
import seaborn as sns

# Import regex module for text cleaning
import re

# Import combinations for skill pair generation
from itertools import combinations

# Import Counter to count skill pairs
from collections import Counter

# Set seaborn style for better visuals
sns.set_style("whitegrid")

# Set default figure size for all plots
plt.rcParams["figure.figsize"] = (10,6)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### **Step 2:** Inspecting the data

In [ ]:
# Read Excel file
df = pd.read_excel('/kaggle/input/indian-job-market-dataset-2025-2026/indian-job-market-dataset-2025.xlsx')

# Show first five rows
print(df.head())

In [ ]:
df.shape

### **Step 3:** Data Cleaning

In [ ]:
# Remove leading and trailing spaces from column names
df.columns = df.columns.str.strip()

# Drop rows where job title OR skills are missing (null)
df = df.dropna(subset=["title", "tagsAndSkills"])

# Convert job titles to lowercase for consistent grouping
df["title"] = df["title"].str.lower()

# Convert skills text to lowercase for uniformity
df["tagsAndSkills"] = df["tagsAndSkills"].str.lower()

# Convert location to lowercase
df["location"] = df["location"].str.lower()

# Check dataset shape after cleaning
df.shape


### Step 4: Feature Engineering(Experience and Salary)

In [ ]:
# Calculate average experience using min and max columns
df["avg_experience"] = (df["minimumExperience"] + df["maximumExperience"]) / 2

# Calculate average salary using min and max columns
df["avg_salary"] = (df["minimumSalary"] + df["maximumSalary"]) / 2

# Convert salary column to numeric
#error='coerce' means if any value cannot be converted to a number,replace it with Nan(Missing value) insted of giving error
df["avg_salary"] = pd.to_numeric(df["avg_salary"], errors="coerce")

# Convert experience column to numeric
df["avg_experience"] = pd.to_numeric(df["avg_experience"], errors="coerce")

# View summary statistics
df[["avg_experience", "avg_salary"]].describe()


### **Step 5:** Intelligent Role Classification

In [ ]:
df["title"].value_counts()

### **Step6:** Clean Title

In [ ]:
# Make a copy to preserve original title
df["clean_title"] = df["title"]

# Convert to lowercase
df["clean_title"] = df["clean_title"].str.lower()

# Remove text inside brackets (experience, wfh, etc.)
df["clean_title"] = df["clean_title"].str.replace(r"\(.*?\)", "", regex=True)

# Remove content after hyphen (usually location or extra info)
df["clean_title"] = df["clean_title"].str.split("-").str[0]

# Remove newline characters
df["clean_title"] = df["clean_title"].str.replace("\n", " ", regex=False)

# Remove multiple spaces
df["clean_title"] = df["clean_title"].str.replace(r"\s+", " ", regex=True)

# Remove non-alphabet characters
df["clean_title"] = df["clean_title"].str.replace(r"[^a-zA-Z ]", "", regex=True)

# Strip leading/trailing spaces
df["clean_title"] = df["clean_title"].str.strip()

# Check result
df[["title", "clean_title"]].head(5)


### **Step7:** Extract Core Designation

In [ ]:
def normalize_designation(title):
    
    # Data roles
    if "data scientist" in title:
        return "data scientist"
    elif "data analyst" in title:
        return "data analyst"
    
    # Software roles
    elif "software development engineer" in title:
        return "software development engineer"
    elif "application developer" in title:
        return "application developer"
    elif "developer" in title:
        return "developer"
    
    # Sales roles
    elif "sales executive" in title:
        return "sales executive"
    elif "business development executive" in title:
        return "business development executive"
    
    # Healthcare
    elif "pharmacist" in title:
        return "pharmacist"
    elif "doctor" in title:
        return "doctor"
    
    # Management
    elif "manager" in title:
        return "manager"
    elif "lead" in title:
        return "lead"
    
    else:
        return title  # fallback

# Apply normalization
df["normalized_title"] = df["clean_title"].apply(normalize_designation)

df["normalized_title"].value_counts().head(10)


### **Step8:** Role Group Classification based on Normalized Title

In [ ]:
def classify_role(title):
    
    if any(x in title for x in ["data scientist", "data analyst"]):
        return "Data Science"
    
    elif any(x in title for x in ["developer", "engineer", "application"]):
        return "Software Engineering"
    
    elif any(x in title for x in ["sales", "business development"]):
        return "Sales"
    
    elif any(x in title for x in ["pharmacist", "doctor", "nurse", "medical"]):
        return "Healthcare"
    
    elif any(x in title for x in ["account", "finance", "billing", "tax"]):
        return "Finance"

    # Marketing related roles
    elif any(x in title for x in ["marketing", "seo", "growth"]):
        return "Marketing"
    
    elif any(x in title for x in ["hr", "recruit"]):
        return "Human Resources"
    
    elif any(x in title for x in ["manager", "lead", "head"]):
        return "Management"
    
    else:
        return "Other"

# Apply role classification
df["role_group"] = df["normalized_title"].apply(classify_role)

df["role_group"].value_counts()


In [ ]:
print("Unique titles before cleaning:", df["title"].nunique())
print("Unique titles after normalization:", df["normalized_title"].nunique())


### **Step 9**: Role Distribution Visualization

In [ ]:
# Count number of jobs per role group
role_counts = df["role_group"].value_counts()

# Create barplot showing distribution
sns.barplot(x=role_counts.index, y=role_counts.values)

# Rotate x labels for better readability
plt.xticks(rotation=45)

# Add title
plt.title("Job Distribution by Role Group")

# Show plot
plt.show()


### **Step 10:** Salary Distribution by Role

In [ ]:
role_counts = df["role_group"].value_counts()

plt.figure(figsize=(8,8))

plt.pie(
    role_counts,
    labels=role_counts.index,
    autopct="%1.1f%%",
    startangle=140
)

plt.title("Job Distribution by Role")
plt.show()




### **Step 10**: Experience vs Salary Relationship

In [ ]:
# Scatterplot to observe correlation between experience and salary
sns.scatterplot(
    data=df,
    x="avg_experience",
    y="avg_salary",
    hue="role_group",  # Color by role
    alpha=0.6          # Slight transparency
)

# Add title
plt.title("Experience vs Salary by Role")

# Show plot
plt.show()


In [ ]:
# Identify top 10 locations by job count
top_cities = df["location"].value_counts().head(10)

# Plot horizontal bar chart
sns.barplot(x=top_cities.values, y=top_cities.index)

# Add title
plt.title("Top 10 Hiring Locations")

# Show plot
plt.show()


In [ ]:
# Remove rows where rating is missing
df = df.dropna(subset=["AggregateRating"])

# Convert rating to numeric (very important)
df["AggregateRating"] = pd.to_numeric(df["AggregateRating"], errors="coerce")

# Create rating bands
df["rating_band"] = pd.cut(
    df["AggregateRating"],
    bins=[0, 2, 3, 4, 5],
    labels=["Low (0-2)", "Below Avg (2-3)", "Good (3-4)", "Excellent (4-5)"],
    include_lowest=True
)

# Check if created
df[["AggregateRating", "rating_band"]].head()


In [ ]:
# Calculate average salary per rating band
rating_salary = (
    df.groupby("rating_band")["avg_salary"]
    .mean()
    .reset_index()
)

# Line plot
sns.lineplot(
    data=rating_salary,
    x="rating_band",
    y="avg_salary",
    marker="o"
)

plt.title("Average Salary Trend Across Company Ratings")
plt.xticks(rotation=30)
plt.show()
